In [1]:
# %load_ext cudf.pandas

In [2]:
# %load_ext cudf.pandas
# Import Visualization FrameWork Tools
import pandas as pd
import numpy as np

# import matplotlib.pyplot as plt
# import seaborn as sns
# # import missingno as msno
# import matplotlib.pyplot as plt
# from PIL import Image
# import matplotlib.patches as mpatches
import time

In [3]:
start_time = time.time()

In [4]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/new_notebooks/nyc-airbnb/AB_NYC_2019.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [5]:
### cell 1 ###

df.shape

(48895, 16)

In [6]:
### cell 2 ###

factor = 10
df = pd.concat([df] * factor)

In [7]:
### cell 3 ###

# check null-value and each columns' dtype to do EDA.
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 488950 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              488950 non-null  int64  
 1   name                            488790 non-null  object 
 2   host_id                         488950 non-null  int64  
 3   host_name                       488740 non-null  object 
 4   neighbourhood_group             488950 non-null  object 
 5   neighbourhood                   488950 non-null  object 
 6   latitude                        488950 non-null  float64
 7   longitude                       488950 non-null  float64
 8   room_type                       488950 non-null  object 
 9   price                           488950 non-null  int64  
 10  minimum_nights                  488950 non-null  int64  
 11  number_of_reviews               488950 non-null  int64  
 12  last_review      

In [8]:
### cell 4 ###

df = df.drop(["id", "last_review", "name", "host_name"], axis=1)
df.head()

,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,1,365
3,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0


In [9]:
### cell 5 ###

df = df.drop(df[df["availability_365"] == 0].index, axis=0)
df = df.reset_index()
df.head()

,index,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,0,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,1,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,2,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,1,365
3,3,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,5,7322,Manhattan,Murray Hill,40.74767,-73.97500,Entire home/apt,200,3,74,0.59,1,129


In [10]:
### cell 6 ###

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df.head()

,index,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,0,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,1,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,2,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365
3,3,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,5,7322,Manhattan,Murray Hill,40.74767,-73.97500,Entire home/apt,200,3,74,0.59,1,129


In [11]:
### cell 7 ###

df.isnull().sum()

index                             0
host_id                           0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [12]:
### cell 8 ###

df.shape

(313620, 13)

In [13]:
# def eda_map(df,feature, ax):
#     plt.figure(figsize=(15,15))
#     image = Image.open("/kaggle/input/new-york-city-airbnb-open-data/New_York_City_.png") # Load the image
#     df.plot(kind='scatter', x='longitude', y='latitude', label='AirBnB On service', c=str(feature), ax=ax, # Create the scatter plot
#             cmap=plt.get_cmap('jet'), colorbar=True, alpha=0.4, zorder=5, s=1)
#     ax.imshow(image,extent=[-74.258, -73.7, 40.49,40.92]) # Set the extent of the image

In [14]:
# plt.figure(figsize=(20,15))
# ax = plt.gca()
# eda_map(df,'price',ax)

In [15]:
### cell 9 ###

nyc_sub_1 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

x_1 = nyc_sub_1
y_1 = []
for i_1 in x_1:
    y_1.append(df[df["neighbourhood_group"] == i_1]["price"].values.tolist())
y_1

[[225,
  150,
  200,
  79,
  150,
  135,
  85,
  85,
  140,
  190,
  150,
  44,
  180,
  50,
  52,
  50,
  40,
  68,
  135,
  150,
  151,
  200,
  110,
  69,
  180,
  375,
  250,
  52,
  80,
  99,
  225,
  230,
  51,
  65,
  190,
  200,
  150,
  110,
  285,
  94,
  60,
  140,
  89,
  98,
  60,
  65,
  500,
  100,
  59,
  140,
  350,
  199,
  235,
  225,
  170,
  170,
  150,
  85,
  120,
  185,
  105,
  130,
  115,
  195,
  156,
  69,
  225,
  219,
  99,
  250,
  125,
  170,
  165,
  150,
  99,
  75,
  275,
  299,
  83,
  123,
  105,
  200,
  121,
  100,
  130,
  199,
  130,
  64,
  250,
  239,
  305,
  155,
  60,
  120,
  150,
  250,
  500,
  225,
  125,
  92,
  175,
  99,
  140,
  500,
  120,
  110,
  130,
  175,
  205,
  390,
  75,
  90,
  68,
  115,
  129,
  75,
  212,
  95,
  135,
  124,
  122,
  109,
  85,
  195,
  575,
  150,
  90,
  65,
  500,
  250,
  125,
  200,
  113,
  250,
  150,
  150,
  71,
  169,
  139,
  95,
  150,
  125,
  130,
  67,
  212,
  249,
  60,
  120,
  169,
 

In [16]:
### cell 10 ###

nyc_sub_2 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
# --- End of sample data creation ---


# The optimized one-liner using groupby
# .apply(list) collects all values for each group into a list
grouped_prices = df.groupby("neighbourhood_group")["price"].apply(list)

# The result is a pandas Series, with boroughs as the index
# and the list of prices as the values.
# print(grouped_prices)
#
# Output would look like:
# neighbourhood_group
# Bronx              [175, 229, 403, 114, 308, 401, 381, 169, 14...
# Brooklyn           [187, 134, 421, 112, 179, 344, 185, 335, 41...
# Manhattan          [376, 269, 396, 429, 332, 287, 237, 275, 43...
# Queens             [487, 310, 342, 290, 488, 303, 442, 334, 16...
# Staten Island      [410, 423, 222, 172, 497, 180, 290, 391, 16...

# To get the exact same output as your original code (a list of lists in a specific order)
# you can reindex the result and convert it to a list.
y_optimized = grouped_prices.reindex(nyc_sub_2).tolist()
y_optimized

[[225,
  150,
  200,
  79,
  150,
  135,
  85,
  85,
  140,
  190,
  150,
  44,
  180,
  50,
  52,
  50,
  40,
  68,
  135,
  150,
  151,
  200,
  110,
  69,
  180,
  375,
  250,
  52,
  80,
  99,
  225,
  230,
  51,
  65,
  190,
  200,
  150,
  110,
  285,
  94,
  60,
  140,
  89,
  98,
  60,
  65,
  500,
  100,
  59,
  140,
  350,
  199,
  235,
  225,
  170,
  170,
  150,
  85,
  120,
  185,
  105,
  130,
  115,
  195,
  156,
  69,
  225,
  219,
  99,
  250,
  125,
  170,
  165,
  150,
  99,
  75,
  275,
  299,
  83,
  123,
  105,
  200,
  121,
  100,
  130,
  199,
  130,
  64,
  250,
  239,
  305,
  155,
  60,
  120,
  150,
  250,
  500,
  225,
  125,
  92,
  175,
  99,
  140,
  500,
  120,
  110,
  130,
  175,
  205,
  390,
  75,
  90,
  68,
  115,
  129,
  75,
  212,
  95,
  135,
  124,
  122,
  109,
  85,
  195,
  575,
  150,
  90,
  65,
  500,
  250,
  125,
  200,
  113,
  250,
  150,
  150,
  71,
  169,
  139,
  95,
  150,
  125,
  130,
  67,
  212,
  249,
  60,
  120,
  169,
 

In [17]:
### cell 11 ###

df["price"].values[0]

149

In [18]:
### cell 12 ###

# Seperate by each Price
df["price_group"] = 0
for idx, pri in enumerate(df["price"]):
    if df.loc[idx, "price"] < 50:
        df.loc[idx, "price_group"] = 0
    elif 50 <= df.loc[idx, "price"] < 250:
        df.loc[idx, "price_group"] = 1
    elif 250 <= df.loc[idx, "price"] < 500:
        df.loc[idx, "price_group"] = 2
    elif 500 <= df.loc[idx, "price"] < 750:
        df.loc[idx, "price_group"] = 3
    elif 750 <= df.loc[idx, "price"] < 1000:
        df.loc[idx, "price_group"] = 4
    else:
        df.loc[idx, "price_group"] = 5

In [19]:
### cell 13 ###

suburb_area = [59100000, 183400000, 464900000, 109000000, 151500000]
distance_room_to_room = []
for i_2, city_1 in enumerate(nyc_sub_2):
    distance_room_to_room.append(
        (suburb_area[i_2] / len(df[df["neighbourhood_group"] == city_1])) * (1 / 1000)
    )

distance_room_to_room

[0.43587285198023457,
 1.495921696574225,
 10.816658911121452,
 11.925601750547047,
 45.770392749244714]

In [20]:
### cell 14 ###

total_amount_sub = []
total_amount_sub_50_250 = []
aver_price_total = []
aver_price_50_250 = []
for i_3, city_2 in enumerate(nyc_sub_2):
    req = df[
        (df["neighbourhood_group"] == city_2) & (df["price"] > 50) & (df["price"] < 250)
    ]
    total_amount_sub.append(len(df[df["neighbourhood_group"] == city_2]))
    total_amount_sub_50_250.append(len(req))
    aver_price_total.append(
        round(np.mean(df[df["neighbourhood_group"] == city_2]["price"]), 3)
    )
    aver_price_50_250.append(round(np.mean(req["price"]), 3))
total_amount_sub, total_amount_sub_50_250, aver_price_total, aver_price_50_250

([135590, 122600, 42980, 9140, 3310],
 [98450, 92400, 29720, 5730, 2250],
 [214.202, 132.852, 100.03, 89.008, 114.23],
 [137.576, 113.951, 99.322, 92.529, 99.213])

In [21]:
### cell 15 ###

nei_x_1 = []
nei_y_1 = []
colors_1 = []

for nei_1 in df["neighbourhood"].value_counts().index:
    nei_x_1.append(nei_1)
    nei_y_1.append(np.mean(df[df["neighbourhood"] == nei_1]["price"].values))

for neigh_1 in nei_x_1:
    if (
        df[df["neighbourhood"] == neigh_1]["neighbourhood_group"].values[0]
        == "Manhattan"
    ):
        colors_1.append("red")
    elif (
        df[df["neighbourhood"] == neigh_1]["neighbourhood_group"].values[0]
        == "Brooklyn"
    ):
        colors_1.append("blue")
    elif (
        df[df["neighbourhood"] == neigh_1]["neighbourhood_group"].values[0] == "Queens"
    ):
        colors_1.append("green")
    elif df[df["neighbourhood"] == neigh_1]["neighbourhood_group"].values[0] == "Bronx":
        colors_1.append("orange")
    elif (
        df[df["neighbourhood"] == neigh_1]["neighbourhood_group"].values[0]
        == "Staten Island"
    ):
        colors_1.append("purple")

nei_x_1, nei_y_1, colors_1

(['Bedford-Stuyvesant',
  'Williamsburg',
  'Harlem',
  'Bushwick',
  "Hell's Kitchen",
  'Upper East Side',
  'Upper West Side',
  'Midtown',
  'East Village',
  'Crown Heights',
  'East Harlem',
  'Chelsea',
  'Greenpoint',
  'Financial District',
  'Washington Heights',
  'Astoria',
  'Lower East Side',
  'East Flatbush',
  'West Village',
  'Flushing',
  'Flatbush',
  'Murray Hill',
  'Long Island City',
  'Prospect-Lefferts Gardens',
  'Clinton Hill',
  'Park Slope',
  'Fort Greene',
  'Kips Bay',
  'Ridgewood',
  'Sunset Park',
  'Sunnyside',
  'SoHo',
  'Theater District',
  'Chinatown',
  'East New York',
  'Jamaica',
  'Ditmars Steinway',
  'Greenwich Village',
  'Gramercy',
  'Prospect Heights',
  'Elmhurst',
  'Woodside',
  'South Slope',
  'East Elmhurst',
  'Inwood',
  'Gowanus',
  'Jackson Heights',
  'Canarsie',
  'Sheepshead Bay',
  'Nolita',
  'Morningside Heights',
  'Carroll Gardens',
  'Tribeca',
  'Cypress Hills',
  'Borough Park',
  'Bay Ridge',
  'Kensington',
  

In [22]:
### cell 16 ###

nei_x_aver_h_1 = []
nei_x_aver_l_1 = []
nei_y_aver_h_1 = []
nei_y_aver_l_1 = []
for nei_2 in df["neighbourhood"].value_counts().index:
    if np.mean(df[df["neighbourhood"] == nei_2]["price"].values) >= round(
        np.mean(nei_y_1), 3
    ):
        nei_x_aver_h_1.append(nei_2)
        nei_y_aver_h_1.append(np.mean(df[df["neighbourhood"] == nei_2]["price"].values))
    elif np.mean(df[df["neighbourhood"] == nei_2]["price"].values) < round(
        np.mean(nei_y_1), 3
    ):
        nei_x_aver_l_1.append(nei_2)
        nei_y_aver_l_1.append(np.mean(df[df["neighbourhood"] == nei_2]["price"].values))

nei_x_aver_h_1, nei_x_aver_l_1, nei_y_aver_h_1, nei_y_aver_l_1

(['Williamsburg',
  "Hell's Kitchen",
  'Upper East Side',
  'Upper West Side',
  'Midtown',
  'East Village',
  'Chelsea',
  'Greenpoint',
  'Financial District',
  'Lower East Side',
  'West Village',
  'Murray Hill',
  'Clinton Hill',
  'Park Slope',
  'Fort Greene',
  'Kips Bay',
  'SoHo',
  'Theater District',
  'Chinatown',
  'Greenwich Village',
  'Gramercy',
  'Prospect Heights',
  'South Slope',
  'Gowanus',
  'Nolita',
  'Carroll Gardens',
  'Tribeca',
  'Bay Ridge',
  'Boerum Hill',
  'Windsor Terrace',
  'Brooklyn Heights',
  'Little Italy',
  'Arverne',
  'Brighton Beach',
  'Cobble Hill',
  'NoHo',
  'Red Hook',
  'Flatiron District',
  'Downtown Brooklyn',
  'Bayside',
  'Roosevelt Island',
  'Battery Park City',
  'Civic Center',
  'Columbia St',
  'Far Rockaway',
  'Vinegar Hill',
  'DUMBO',
  'Randall Manor',
  'Jamaica Estates',
  'City Island',
  'Coney Island',
  'Eastchester',
  'Belle Harbor',
  'Riverdale',
  'Unionport',
  'Tottenville',
  'Bay Terrace',
  'Nav

In [23]:
### cell 17 ###

colors_2 = []
for neigh_2 in nei_x_aver_h_1:
    if (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Manhattan"
    ):
        colors_2.append("red")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Brooklyn"
    ):
        colors_2.append("blue")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0] == "Queens"
    ):
        colors_2.append("green")
    elif df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0] == "Bronx":
        colors_2.append("orange")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Staten Island"
    ):
        colors_2.append("purple")
colors_2

['blue',
 'red',
 'red',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'blue',
 'blue',
 'red',
 'red',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'blue',
 'blue',
 'red',
 'blue',
 'red',
 'blue',
 'blue',
 'blue',
 'blue',
 'red',
 'green',
 'blue',
 'blue',
 'red',
 'blue',
 'red',
 'blue',
 'green',
 'red',
 'red',
 'red',
 'blue',
 'green',
 'blue',
 'blue',
 'purple',
 'green',
 'orange',
 'blue',
 'orange',
 'green',
 'orange',
 'orange',
 'purple',
 'green',
 'blue',
 'green',
 'purple',
 'blue',
 'orange',
 'purple',
 'green',
 'purple',
 'green',
 'blue',
 'green',
 'purple',
 'purple',
 'purple']

In [24]:
### cell 18 ###

colors_3 = []
for neigh_3 in nei_x_aver_h_1:
    if (
        df[df["neighbourhood"] == neigh_3]["neighbourhood_group"].values[0]
        == "Manhattan"
    ):
        colors_3.append("red")
    elif (
        df[df["neighbourhood"] == neigh_3]["neighbourhood_group"].values[0]
        == "Brooklyn"
    ):
        colors_3.append("blue")
    elif (
        df[df["neighbourhood"] == neigh_3]["neighbourhood_group"].values[0] == "Queens"
    ):
        colors_3.append("green")
    elif df[df["neighbourhood"] == neigh_3]["neighbourhood_group"].values[0] == "Bronx":
        colors_3.append("orange")
    elif (
        df[df["neighbourhood"] == neigh_3]["neighbourhood_group"].values[0]
        == "Staten Island"
    ):
        colors_3.append("purple")
colors_3

['blue',
 'red',
 'red',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'blue',
 'blue',
 'red',
 'red',
 'red',
 'red',
 'red',
 'red',
 'blue',
 'blue',
 'blue',
 'red',
 'blue',
 'red',
 'blue',
 'blue',
 'blue',
 'blue',
 'red',
 'green',
 'blue',
 'blue',
 'red',
 'blue',
 'red',
 'blue',
 'green',
 'red',
 'red',
 'red',
 'blue',
 'green',
 'blue',
 'blue',
 'purple',
 'green',
 'orange',
 'blue',
 'orange',
 'green',
 'orange',
 'orange',
 'purple',
 'green',
 'blue',
 'green',
 'purple',
 'blue',
 'orange',
 'purple',
 'green',
 'purple',
 'green',
 'blue',
 'green',
 'purple',
 'purple',
 'purple']

In [25]:
### cell 19 ###

x_2 = df["calculated_host_listings_count"].value_counts().sort_index().index.astype(str)
y_2 = df["calculated_host_listings_count"].value_counts().sort_index().values

x_2, y_2

(Index(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13',
        '14', '15', '16', '17', '18', '19', '20', '21', '23', '25', '26', '27',
        '28', '29', '30', '31', '32', '33', '34', '37', '39', '43', '47', '49',
        '50', '52', '65', '87', '91', '96', '103', '121', '232', '327'],
       dtype='object'),
 array([173930,  49220,  24020,  12870,   7590,   5380,   3710,   3840,
          2320,   2060,   1080,   1510,   1280,    580,    700,    160,
           680,    540,    190,    400,    200,    660,    490,    260,
           260,    560,    290,    300,    620,    320,    980,    670,
           210,    380,    430,    470,    980,    500,   1020,    650,
           850,    860,   1880,    920,   1210,   2320,   3270]))

In [26]:
end_time = time.time()
print(end_time - start_time)

74.04992842674255
